**Comparison**

In [5]:
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET
from pathlib import Path

# ============================================================
# SETTINGS
# ============================================================

BASE_DIR = Path(r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre")

SCENARIO_NAME = "weekend_evening"
# options:
# "weekday_morning", "weekday_evening",
# "weekend_morning", "weekend_evening"

REAL_FILE = BASE_DIR / rf"detector_data_aktuell\representative_day_routesampler_base_warmup15\{SCENARIO_NAME}_detector_15min_counts_analysis_only.csv"
SIM_E1_FILE = BASE_DIR / r"digital_twin\simulation\Corridor_e1_output.xml"

OUT_DIR = BASE_DIR / r"digital_twin\validation_outputs_new"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# warm-up ignored -> compare only 900-4500
SIM_BEGINS = [900, 1800, 2700, 3600]
SIM_ENDS = [1800, 2700, 3600, 4500]

# ============================================================
# SUMO DETECTOR ID -> (intersection, detector_id)
# ============================================================

SUMO_ID_MAP = {
    # ---------------- LSA10 ----------------
    "LSA10_E_id3":   ("LD-LSA10", 3),
    "LSA10_E_id4":   ("LD-LSA10", 4),
    "LSA10_NE_id29": ("LD-LSA10", 29),
    "LSA10_NW_id2":  ("LD-LSA10", 2),
    "LSA10_NW_id26": ("LD-LSA10", 26),
    "LSA10_NW_id27": ("LD-LSA10", 27),
    "LSA10_S_id31":  ("LD-LSA10", 31),
    "LSA10_W_id1":   ("LD-LSA10", 1),

    # ---------------- LSA16 ----------------
    "LSA16_E_id10":    ("LD-LSA16", 10),
    "LSA16_E_id11":    ("LD-LSA16", 11),
    "LSA16_E_id14":    ("LD-LSA16", 14),
    "LSA16_Eout_id13": ("LD-LSA16", 13),
    "LSA16_N_id6":     ("LD-LSA16", 6),
    "LSA16_N_id7":     ("LD-LSA16", 7),
    "LSA16_S_id1":     ("LD-LSA16", 1),
    "LSA16_S_id2":     ("LD-LSA16", 2),
    "LSA16_S_id33":    ("LD-LSA16", 33),
    "LSA16_W_id3":     ("LD-LSA16", 3),
    "LSA16_W_id4":     ("LD-LSA16", 4),
    "LSA16_W_id5":     ("LD-LSA16", 5),
    "LSA16_Wout_id15": ("LD-LSA16", 15),

    # ---------------- LSA1 ----------------
    "LSA1_E_id13": ("LD-LSA1", 13),
    "LSA1_E_id14": ("LD-LSA1", 14),
    "LSA1_E_id23": ("LD-LSA1", 23),
    "LSA1_E_id24": ("LD-LSA1", 24),
    "LSA1_E_id25": ("LD-LSA1", 25),
    "LSA1_E_id26": ("LD-LSA1", 26),
    "LSA1_E_id27": ("LD-LSA1", 27),
    "LSA1_N_id28": ("LD-LSA1", 28),
    "LSA1_N_id30": ("LD-LSA1", 30),
    "LSA1_N_id39": ("LD-LSA1", 39),
    "LSA1_N_id40": ("LD-LSA1", 40),
    "LSA1_S_id11": ("LD-LSA1", 11),
    "LSA1_S_id12": ("LD-LSA1", 12),
    "LSA1_W_id10": ("LD-LSA1", 10),
    "LSA1_W_id7":  ("LD-LSA1", 7),
    "LSA1_W_id8":  ("LD-LSA1", 8),
    "LSA1_W_id9":  ("LD-LSA1", 9),

    # ---------------- LSA9 ----------------
    "LSA9_E_id8":  ("LD-LSA9", 8),
    "LSA9_E_id9":  ("LD-LSA9", 9),
    "LSA9_WE_id4": ("LD-LSA9", 4),
    "LSA9_WE_id5": ("LD-LSA9", 5),
    "LSA9_WE_id6": ("LD-LSA9", 6),
    "LSA9_WE_id7": ("LD-LSA9", 7),
    "LSA9_W_id1":  ("LD-LSA9", 1),
    "LSA9_W_id2":  ("LD-LSA9", 2),
    "LSA9_S_id10": ("LD-LSA9", 10),
}

# ============================================================
# VALIDATION DETECTORS
# ============================================================

VALIDATION_KEEP = {
    # LSA16
    ("LD-LSA16", 1), ("LD-LSA16", 2), ("LD-LSA16", 3), ("LD-LSA16", 4),
    ("LD-LSA16", 6), ("LD-LSA16", 7), ("LD-LSA16", 10), ("LD-LSA16", 11),
    ("LD-LSA16", 13), ("LD-LSA16", 15), ("LD-LSA16", 33),

    # LSA10
    ("LD-LSA10", 2), ("LD-LSA10", 3), ("LD-LSA10", 4),
    ("LD-LSA10", 26), ("LD-LSA10", 27), ("LD-LSA10", 29), ("LD-LSA10", 31),

    # LSA1
    ("LD-LSA1", 7), ("LD-LSA1", 9), ("LD-LSA1", 13), ("LD-LSA1", 14),
    ("LD-LSA1", 23), ("LD-LSA1", 24), ("LD-LSA1", 26), ("LD-LSA1", 27),
    ("LD-LSA1", 28), ("LD-LSA1", 30), ("LD-LSA1", 40),

    # LSA9
    ("LD-LSA9", 1), ("LD-LSA9", 2), ("LD-LSA9", 4), ("LD-LSA9", 5),
    ("LD-LSA9", 6), ("LD-LSA9", 7), ("LD-LSA9", 9), ("LD-LSA9", 10),
}

EXCLUDE_VALIDATION = {
    ("LD-LSA1", 11),
    ("LD-LSA1", 12),
    ("LD-LSA1", 25),
    # gerekirse buraya ekle
}

# ============================================================
# LSA16 APPROACH GROUPS
# ============================================================

LSA16_APPROACH_GROUPS = {
    "west_approach": [3, 4],
    "south_approach": [1, 2, 33],
    "east_approach": [10, 11],
    "north_approach": [6, 7],
    "exits": [13, 15],
}

LSA16_DETECTOR_LABELS = {
    1: "south_id1",
    2: "south_id2",
    3: "west_id3",
    4: "west_id4",
    6: "north_id6",
    7: "north_id7",
    10: "east_id10",
    11: "east_id11",
    13: "east_out_id13",
    15: "west_out_id15",
    33: "south_id33",
}

# ============================================================
# METRICS
# ============================================================

def corr_from_series(sim, obs):
    sim = np.asarray(sim, dtype=float)
    obs = np.asarray(obs, dtype=float)
    if len(sim) < 2:
        return np.nan
    if np.all(sim == sim[0]) or np.all(obs == obs[0]):
        return np.nan
    return np.corrcoef(sim, obs)[0, 1]

# ============================================================
# READ REAL 15-MIN DATA
# ============================================================

real_df = pd.read_csv(REAL_FILE)

real_df["detector_id"] = pd.to_numeric(real_df["detector_id"], errors="coerce").astype("Int64")
real_df["mean_occupancy"] = pd.to_numeric(real_df["mean_occupancy"], errors="coerce").fillna(0)
real_df["begin_s"] = pd.to_numeric(real_df["begin_s"], errors="coerce")
real_df["end_s"] = pd.to_numeric(real_df["end_s"], errors="coerce")
real_df["key"] = list(zip(real_df["intersection"], real_df["detector_id"]))

real_df = real_df[
    real_df["key"].isin(VALIDATION_KEEP) &
    (~real_df["key"].isin(EXCLUDE_VALIDATION))
].copy()

real_df["bin_length_s"] = real_df["end_s"] - real_df["begin_s"]
real_df = real_df[real_df["bin_length_s"] == 900].copy()

real_bins = (
    real_df[["begin_s", "end_s"]]
    .drop_duplicates()
    .sort_values(["begin_s", "end_s"])
    .reset_index(drop=True)
)

if len(real_bins) != 4:
    raise ValueError(f"Expected exactly 4 real 15-min bins, found {len(real_bins)}.\n{real_bins}")

real_bins["real_bin_idx"] = range(4)
real_bins["bin_label"] = ["00-15", "15-30", "30-45", "45-60"]

real_agg = (
    real_df.groupby(
        ["intersection", "detector_id", "begin_s", "end_s"],
        as_index=False
    )
    .agg(
        occupancy_real=("mean_occupancy", "mean")
    )
)

real_agg = real_agg.merge(real_bins, on=["begin_s", "end_s"], how="left")
real_agg["bin_idx"] = real_agg["real_bin_idx"]

print("\nREAL BINS USED:")
print(real_bins[["begin_s", "end_s", "bin_label"]].to_string(index=False))

# ============================================================
# READ SIM E1 OUTPUT
# ============================================================

tree = ET.parse(SIM_E1_FILE)
root = tree.getroot()

sim_rows = []
for interval in root.findall("interval"):
    sumo_id = interval.attrib.get("id")
    begin = int(float(interval.attrib.get("begin", 0)))
    end = int(float(interval.attrib.get("end", 0)))
    occupancy = float(interval.attrib.get("occupancy", 0))

    if sumo_id not in SUMO_ID_MAP:
        continue

    intersection, detector_id = SUMO_ID_MAP[sumo_id]
    key = (intersection, detector_id)

    if key not in VALIDATION_KEEP or key in EXCLUDE_VALIDATION:
        continue

    if begin not in SIM_BEGINS or end not in SIM_ENDS:
        continue

    sim_rows.append({
        "intersection": intersection,
        "detector_id": detector_id,
        "bin_idx": SIM_BEGINS.index(begin),
        "sim_begin_s": begin,
        "sim_end_s": end,
        "occupancy_sim": occupancy
    })

sim_df = pd.DataFrame(sim_rows)

if sim_df.empty:
    raise ValueError("Simulation E1 output produced no matching rows.")

sim_agg = (
    sim_df.groupby(
        ["intersection", "detector_id", "bin_idx", "sim_begin_s", "sim_end_s"],
        as_index=False
    )["occupancy_sim"]
    .mean()
)

# ============================================================
# BUILD FULL GRID
# ============================================================

detector_keys = sorted(list(VALIDATION_KEEP - EXCLUDE_VALIDATION))

grid_rows = []
for inter, det in detector_keys:
    for i, (sb, se) in enumerate(zip(SIM_BEGINS, SIM_ENDS)):
        grid_rows.append({
            "intersection": inter,
            "detector_id": det,
            "bin_idx": i,
            "sim_begin_s": sb,
            "sim_end_s": se
        })

grid_df = pd.DataFrame(grid_rows)

real_for_merge = real_agg[[
    "intersection", "detector_id", "bin_idx",
    "begin_s", "end_s", "bin_label", "occupancy_real"
]]

cmp_df = (
    grid_df
    .merge(real_for_merge, on=["intersection", "detector_id", "bin_idx"], how="left")
    .merge(sim_agg, on=["intersection", "detector_id", "bin_idx", "sim_begin_s", "sim_end_s"], how="left")
)

cmp_df = cmp_df.merge(
    real_bins[["real_bin_idx", "begin_s", "end_s", "bin_label"]].rename(columns={"real_bin_idx": "bin_idx"}),
    on="bin_idx",
    how="left",
    suffixes=("", "_fallback")
)

cmp_df["begin_s"] = cmp_df["begin_s"].fillna(cmp_df["begin_s_fallback"])
cmp_df["end_s"] = cmp_df["end_s"].fillna(cmp_df["end_s_fallback"])
cmp_df["bin_label"] = cmp_df["bin_label"].fillna(cmp_df["bin_label_fallback"])

cmp_df = cmp_df.drop(columns=["begin_s_fallback", "end_s_fallback", "bin_label_fallback"])

cmp_df["occupancy_real"] = cmp_df["occupancy_real"].fillna(0)
cmp_df["occupancy_sim"] = cmp_df["occupancy_sim"].fillna(0)

cmp_df["diff_sim_minus_real"] = cmp_df["occupancy_sim"] - cmp_df["occupancy_real"]
cmp_df["abs_error"] = np.abs(cmp_df["diff_sim_minus_real"])
cmp_df["squared_error"] = cmp_df["diff_sim_minus_real"] ** 2

cmp_df = cmp_df.sort_values(["intersection", "detector_id", "bin_idx"]).copy()

# ============================================================
# DETECTOR TOTALS
# ============================================================

detector_totals = (
    cmp_df.groupby(["intersection", "detector_id"], as_index=False)
    .agg(
        mean_real_occupancy=("occupancy_real", "mean"),
        mean_sim_occupancy=("occupancy_sim", "mean"),
        mean_bias=("diff_sim_minus_real", "mean"),
        MAE=("abs_error", "mean"),
        RMSE=("squared_error", lambda s: np.sqrt(np.mean(s))),
        max_abs_error=("abs_error", "max"),
        n_bins=("bin_idx", "count")
    )
    .sort_values(["intersection", "detector_id"])
)

detector_corr_rows = []
for (intersection, detector_id), sub in cmp_df.groupby(["intersection", "detector_id"]):
    detector_corr_rows.append({
        "intersection": intersection,
        "detector_id": detector_id,
        "corr": corr_from_series(sub["occupancy_sim"], sub["occupancy_real"])
    })

detector_corr_df = pd.DataFrame(detector_corr_rows)

detector_totals = detector_totals.merge(
    detector_corr_df,
    on=["intersection", "detector_id"],
    how="left"
)

# ============================================================
# INTERSECTION SUMMARY
# ============================================================

intersection_summary = (
    cmp_df.groupby("intersection", as_index=False)
    .agg(
        mean_real_occupancy=("occupancy_real", "mean"),
        mean_sim_occupancy=("occupancy_sim", "mean"),
        mean_bias=("diff_sim_minus_real", "mean"),
        MAE=("abs_error", "mean"),
        RMSE=("squared_error", lambda s: np.sqrt(np.mean(s))),
        max_abs_error=("abs_error", "max"),
        n_rows=("bin_idx", "count")
    )
    .sort_values("intersection")
)

intersection_corr_rows = []
for intersection, sub in cmp_df.groupby("intersection"):
    intersection_corr_rows.append({
        "intersection": intersection,
        "corr": corr_from_series(sub["occupancy_sim"], sub["occupancy_real"])
    })

intersection_corr_df = pd.DataFrame(intersection_corr_rows)

intersection_summary = intersection_summary.merge(
    intersection_corr_df,
    on="intersection",
    how="left"
)

# ============================================================
# OVERALL SUMMARY
# ============================================================

overall_summary = pd.DataFrame([{
    "scenario": SCENARIO_NAME,
    "mean_real_occupancy": cmp_df["occupancy_real"].mean(),
    "mean_sim_occupancy": cmp_df["occupancy_sim"].mean(),
    "mean_bias": cmp_df["diff_sim_minus_real"].mean(),
    "MAE": cmp_df["abs_error"].mean(),
    "RMSE": np.sqrt(cmp_df["squared_error"].mean()),
    "max_abs_error": cmp_df["abs_error"].max(),
    "corr": corr_from_series(cmp_df["occupancy_sim"], cmp_df["occupancy_real"])
}])

# ============================================================
# SAVE MAIN OUTPUTS
# ============================================================

cmp_out = OUT_DIR / f"{SCENARIO_NAME}_comparison_15min_occupancy_warmup15.csv"
det_out = OUT_DIR / f"{SCENARIO_NAME}_detector_totals_occupancy_warmup15.csv"
int_out = OUT_DIR / f"{SCENARIO_NAME}_intersection_summary_occupancy_warmup15.csv"
ov_out = OUT_DIR / f"{SCENARIO_NAME}_overall_summary_occupancy_warmup15.csv"

cmp_df.to_csv(cmp_out, index=False, encoding="utf-8-sig")
detector_totals.to_csv(det_out, index=False, encoding="utf-8-sig")
intersection_summary.to_csv(int_out, index=False, encoding="utf-8-sig")
overall_summary.to_csv(ov_out, index=False, encoding="utf-8-sig")

# ============================================================
# LSA16 APPROACH-WISE TABLES
# ============================================================

lsa16_df = cmp_df[cmp_df["intersection"] == "LD-LSA16"].copy()
lsa16_df["detector_label"] = lsa16_df["detector_id"].map(LSA16_DETECTOR_LABELS)

for approach_name, det_list in LSA16_APPROACH_GROUPS.items():
    sub = lsa16_df[lsa16_df["detector_id"].isin(det_list)].copy()

    sub = sub[[
        "intersection",
        "detector_id",
        "detector_label",
        "bin_label",
        "begin_s",
        "end_s",
        "sim_begin_s",
        "sim_end_s",
        "occupancy_real",
        "occupancy_sim",
        "diff_sim_minus_real",
        "abs_error"
    ]].sort_values(["detector_id", "begin_s"])

    out_file = OUT_DIR / f"{SCENARIO_NAME}_LSA16_{approach_name}_15min_occupancy_warmup15.csv"
    sub.to_csv(out_file, index=False, encoding="utf-8-sig")

# ============================================================
# PRINT RESULTS
# ============================================================

print("\n" + "=" * 90)
print("OVERALL SUMMARY - 15-MIN OCCUPANCY (WARM-UP 15 MIN IGNORED)")
print("=" * 90)
print(overall_summary.to_string(index=False))

print("\n" + "=" * 90)
print("INTERSECTION SUMMARY - 15-MIN OCCUPANCY (WARM-UP 15 MIN IGNORED)")
print("=" * 90)
print(intersection_summary.to_string(index=False))

print("\n" + "=" * 90)
print("DETECTOR TOTALS - 15-MIN OCCUPANCY (WARM-UP 15 MIN IGNORED)")
print("=" * 90)
print(detector_totals.to_string(index=False))

print("\n" + "=" * 90)
print("15-MIN OCCUPANCY COMPARISON (REAL 4 BINS vs SIM 900-4500)")
print("=" * 90)
print(
    cmp_df[
        [
            "intersection", "detector_id", "bin_label",
            "begin_s", "end_s", "sim_begin_s", "sim_end_s",
            "occupancy_real", "occupancy_sim", "diff_sim_minus_real", "abs_error"
        ]
    ].to_string(index=False)
)

print("\nSaved files:")
print(" -", cmp_out)
print(" -", det_out)
print(" -", int_out)
print(" -", ov_out)


REAL BINS USED:
 begin_s  end_s bin_label
   57600  58500     00-15
   58500  59400     15-30
   59400  60300     30-45
   60300  61200     45-60

OVERALL SUMMARY - 15-MIN OCCUPANCY (WARM-UP 15 MIN IGNORED)
       scenario  mean_real_occupancy  mean_sim_occupancy  mean_bias       MAE      RMSE  max_abs_error     corr
weekend_evening            24.297297           17.958446  -6.338851 12.935743 17.477917          60.78 0.621712

INTERSECTION SUMMARY - 15-MIN OCCUPANCY (WARM-UP 15 MIN IGNORED)
intersection  mean_real_occupancy  mean_sim_occupancy  mean_bias       MAE      RMSE  max_abs_error  n_rows     corr
     LD-LSA1            27.522727           28.699091   1.176364  9.117727 11.182238          27.86      44 0.856941
    LD-LSA10            23.678571            6.860000 -16.818571 17.460714 24.240140          60.78      28 0.306526
    LD-LSA16            19.840909           11.523182  -8.317727 10.485000 13.824899          41.25      44 0.745796
     LD-LSA9            26.531250 